# Expense Claim Analysis

This notebook demonstrates how to create agents that use plugins to process travel expenses from local receipt images, generate an expense claim email, and visualize expense data using a pie chart. Agents dynamically choose functions based on the task context.

Steps:
1. OCR Agent processes the local receipt image and extracts travel expense data.
2. Email Agent generates an expense claim email.

### Example of a travel expense scenario:
Imagine you're an employee traveling for a business meeting in another city. Your company has a policy to reimburse all reasonable travel-related expenses. Here's a breakdown of potential travel expenses:
- Transportation:
Airfare for a round trip from your home city to the destination city.
Taxi or ride-hailing services to and from the airport.
Local transportation in the destination city (like public transit, rental cars, or taxis).

- Accommodation:
Hotel stay for three nights at a mid-range business hotel close to the meeting venue.

- Meals:
Daily meal allowance for breakfast, lunch, and dinner, based on the company's per diem policy.

- Miscellaneous Expenses:
Parking fees at the airport.
Internet access charges at the hotel.
Tips or small service charges.

- Documentation:
You submit all receipts (flights, taxis, hotel, meals, etc.) and a completed expense report for reimbursement.

## Import required libraries

Import the necessary libraries and modules for the notebook.

In [1]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder, Content, Message
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

c:\Users\lujan\Proyectos\Cursos\ai-agents-for-beginners\.venv\Lib\site-packages\agent_framework\_skills.py:122: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Users\lujan\Proyectos\Cursos\ai-agents-for-beginners\.venv\Lib\site-packages\agent_framework\_harness\_file_access.py:602: ExperimentalWarning: [HARNESS] AgentFileStore is experimental and may change or be removed in future versions without notice.


In [2]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Define Expense Models

 Create a Pydantic model for individual expenses and an ExpenseFormatter class to convert a user query into structured expense data.

 Each expense will be represented in the format:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [3]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Defining Tools - Generating the Email

Create a tool function to generate an email for submitting an expense claim.
- This tool uses the `@tool` decorator from the Microsoft Agent Framework.
- It calculates the total amount of the expenses and formats the details into an email body.

In [4]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Loading the Receipt Image

Create a helper function that loads the receipt image as image content, so it can be attached directly to the message sent to the OCR agent (instead of being loaded via a tool call).

> **Why not a tool?** Azure AI Foundry's chat client (`SUPPORTS_RICH_FUNCTION_OUTPUT = False`) does not forward rich content (images) returned from a tool call back to the model — only plain text. So a tool that returns an image is invisible to the model. Attaching the image directly to the input message avoids this limitation.

In [5]:
def load_receipt_image(
    image_path: str = "receipt.jpg",
) -> Content:
    """Load a receipt image as image content the model can see directly for OCR extraction."""
    with open(image_path, "rb") as f:
        image_bytes = f.read()
    return Content.from_data(image_bytes, media_type="image/jpeg")

## Processing Expenses

Define the agents and wire them into a sequential workflow using `WorkflowBuilder`.
- The OCR agent receives the receipt image directly in the input message and extracts structured expense data from it.
- The Email agent takes the extracted data and generates a professional expense claim email using the `generate_expense_email` tool.
- `WorkflowBuilder` with `add_edge` creates a sequential pipeline: OCR Agent → Email Agent.

In [6]:
ocr_agent = client.as_agent(
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "The user message includes the receipt image directly. Analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

Build the sequential workflow and run it to process the receipt image and generate the expense claim email.

> **Note:** The receipt image is now attached directly to the initial `Message` sent to the workflow (as `input_image` content), instead of being returned from a tool call. Azure AI Foundry's client does not forward rich (image) content from tool results back to the model, so routing the image through a tool silently drops it and the OCR agent would see an empty result. Sending the image as part of the user message works with vision-capable models (e.g. `gpt-5-mini`, `gpt-4o`).

In [7]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt_text = (
    "Please extract travel expenses from this receipt image, "
    "focusing on dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)
receipt_image = load_receipt_image("receipt.jpg")
message = Message("user", [prompt_text, receipt_image])

last_author = None
events = workflow.run(
    message,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

C:\Users\lujan\AppData\Local\Temp\ipykernel_15984\406925584.py:3: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build()



# Agent - OCRAgent:
02-May-2022|Penne Alfredo|24.00|Meals;02-May-2022|Chicken Wrap|19.00|Meals;02-May-2022|Jungle Colada|9.00|Meals;02-May-2022|Melon Lime Cooler|9.00|Meals

# Agent - EmailAgent:
Dear Finance Team,

Please find below the details of my expense claim for the business meal on 02-May-2022:

- 02-May-2022 | Penne Alfredo — $24.00 (Meals)
- 02-May-2022 | Chicken Wrap — $19.00 (Meals)
- 02-May-2022 | Jungle Colada — $9.00 (Meals)
- 02-May-2022 | Melon Lime Cooler — $9.00 (Meals)

Subtotal: $61.00
Service charge and tax were included on the receipt; total paid shown: $75.15.

Please reimburse the subtotal of $61.00 to my usual payroll/bank details. Receipts are attached.

If you need any additional information, let me know.

Thanks,
[Your Name]